In [1]:

# ============================================================
# CrisisMMD - Logistic Regression & Bootstrap CI
# Original experiment + Additional Fusion vs Image-only CI
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler

SPLITS = Path("../data/splits") / "crisismmd_datasplit_agreed_label"
CACHE = Path("../cache")
C_GRID = [0.001, 0.01, 0.1, 1.0]
RNG = np.random.default_rng(0)


# ============================================================
# 1. Load cached features (ORIGINAL)
# ============================================================

def load_features():
    """Image features keyed by image_id and text by tweet_id."""
    img = np.load(CACHE / "image_b0.npz", allow_pickle=True)
    txt = np.load(CACHE / "text_bert.npz", allow_pickle=True)

    img_map = dict(zip(img["image_id"], range(len(img["image_id"]))))
    txt_map = dict(zip(txt["tweet_id"], range(len(txt["tweet_id"]))))

    return (
        img["feat"].astype(np.float32), img_map,
        txt["feat"].astype(np.float32), txt_map
    )


def load_task(task, feats):
    """Load train, dev and test datasets."""
    img, img_map, txt, txt_map = feats
    out = {}

    for split in ["train", "dev", "test"]:
        df = pd.read_csv(
            SPLITS / f"task_{task}_text_img_agreed_lab_{split}.tsv",
            sep="\t", dtype=str, quoting=3
        )

        out[split] = {
            "image": img[[img_map[i] for i in df.image_id]],
            "text": txt[[txt_map[i] for i in df.tweet_id]],
            "y": df.label.values
        }

    return out


# ============================================================
# 2. Model training (ORIGINAL)
# ============================================================

def fit_best(x_train, y_train, x_dev, y_dev):
    """Select logistic regression C using dev macro-F1."""
    scaler = StandardScaler().fit(x_train)

    xt = scaler.transform(x_train)
    xd = scaler.transform(x_dev)

    best = None

    for c in C_GRID:
        model = LogisticRegression(
            C=c,
            max_iter=3000,
            class_weight="balanced"
        ).fit(xt, y_train)

        score = f1_score(
            y_dev, model.predict(xd), average="macro"
        )

        if best is None or score > best[0]:
            best = (score, c, model)

    return scaler, best[2], best[1]


def scores(y, pred):
    """Accuracy, macro-F1 and weighted F1."""
    return (
        accuracy_score(y, pred),
        f1_score(y, pred, average="macro"),
        f1_score(y, pred, average="weighted")
    )


# ============================================================
# 3. Bootstrap function (ORIGINAL - UNCHANGED)
# ============================================================

def bootstrap_diff(y, pred_a, pred_b, n=1000):
    """95% CI for macro-F1 difference using paired bootstrap."""

    idx = RNG.integers(0, len(y), size=(n, len(y)))

    diffs = [
        f1_score(y[i], pred_a[i], average="macro")
        - f1_score(y[i], pred_b[i], average="macro")
        for i in idx
    ]

    return np.percentile(diffs, [2.5, 97.5])


# ============================================================
# 4. Run experiment (ORIGINAL + NEW CI)
# ============================================================

def run(task, feats):
    """Train and compare image, text, early and late fusion."""

    d = load_task(task, feats)

    for s in d.values():
        s["early"] = np.hstack([s["image"], s["text"]])

    rows, preds, probs = [], {}, {}

    # Original: Train image, text, early fusion
    for name in ["image", "text", "early"]:

        scaler, model, c = fit_best(
            d["train"][name],
            d["train"]["y"],
            d["dev"][name],
            d["dev"]["y"]
        )

        for split in ["train", "dev", "test"]:
            x = scaler.transform(d[split][name])
            probs[name, split] = model.predict_proba(x)
            preds[name, split] = model.predict(x)

        classes = model.classes_
        rows.append([name, c])

    # Original: Late fusion
    best_w = max(
        np.linspace(0, 1, 11),
        key=lambda w: f1_score(
            d["dev"]["y"],
            classes[
                (
                    w * probs["image", "dev"]
                    + (1 - w) * probs["text", "dev"]
                ).argmax(1)
            ],
            average="macro"
        )
    )

    for split in ["train", "dev", "test"]:
        p = (
            best_w * probs["image", split]
            + (1 - best_w) * probs["text", split]
        )
        preds["late", split] = classes[p.argmax(1)]

    rows.append(["late", f"w_image={best_w:.1f}"])

    # Original: Model evaluation table
    table = []

    for name, setting in rows:
        acc, mf1, wf1 = scores(
            d["test"]["y"], preds[name, "test"]
        )

        tr = f1_score(
            d["train"]["y"],
            preds[name, "train"],
            average="macro"
        )

        dv = f1_score(
            d["dev"]["y"],
            preds[name, "dev"],
            average="macro"
        )

        table.append([
            name, setting,
            round(tr, 3),
            round(dv, 3),
            round(acc, 3),
            round(mf1, 3),
            round(wf1, 3)
        ])

    print(
        f"\n=== {task} (agreed labels) "
        f"train {len(d['train']['y'])}, "
        f"dev {len(d['dev']['y'])}, "
        f"test {len(d['test']['y'])} ==="
    )

    print(pd.DataFrame(
        table,
        columns=[
            "model", "setting", "train mF1",
            "dev mF1", "test acc",
            "test mF1", "test wF1"
        ]
    ).to_string(index=False))

    # ========================================================
    # ORIGINAL: CI vs best unimodal selected on dev
    # ========================================================

    y = d["test"]["y"]

    uni = max(
        ["image", "text"],
        key=lambda m: f1_score(
            d["dev"]["y"],
            preds[m, "dev"],
            average="macro"
        )
    )

    print("\n--- Original Bootstrap CI ---")

    for fused in ["early", "late"]:
        lo, hi = bootstrap_diff(
            y,
            preds[fused, "test"],
            preds[uni, "test"]
        )

        print(
            f"{fused} minus {uni} test macro-F1, "
            f"95% bootstrap interval [{lo:.3f}, {hi:.3f}]"
        )

    # ========================================================
    # NEW: Additional Bootstrap CI vs Image-only
    # Only for 5-class humanitarian task
    # ========================================================

    if task == "humanitarian":

        print("\n--- NEW: Fusion vs Image-only ---")

        image_f1 = f1_score(
            y,
            preds["image", "test"],
            average="macro"
        )

        for fused in ["early", "late"]:

            fusion_f1 = f1_score(
                y,
                preds[fused, "test"],
                average="macro"
            )

            difference = fusion_f1 - image_f1

            lo, hi = bootstrap_diff(
                y,
                preds[fused, "test"],
                preds["image", "test"],
                n=1000
            )

            print(f"\n{fused.upper()} vs IMAGE-ONLY")
            print(f"Macro-F1 difference: {difference:.3f}")
            print(f"95% Bootstrap CI: [{lo:.3f}, {hi:.3f}]")

            if lo > 0:
                print("CI excludes zero: positive improvement")
            else:
                print("CI does not establish positive improvement")

    # ========================================================
    # ORIGINAL: Per-class F1
    # ========================================================

    print("\nper-class test F1")

    per = {
        m: f1_score(
            y,
            preds[m, "test"],
            average=None,
            labels=classes
        )
        for m in ["image", "text", "early", "late"]
    }

    print(
        pd.DataFrame(per, index=classes)
        .round(3)
        .to_string()
    )


# ============================================================
# 5. Execute experiment (ORIGINAL)
# ============================================================

feats = load_features()

for task in ["informative", "humanitarian"]:
    run(task, feats)



=== informative (agreed labels) train 9601, dev 1573, test 1534 ===
model     setting  train mF1  dev mF1  test acc  test mF1  test wF1
image       0.001      0.852    0.806     0.842     0.826     0.844
 text       0.001      0.833    0.819     0.845     0.830     0.847
early       0.001      0.916    0.876     0.901     0.890     0.902
 late w_image=0.5      0.906    0.885     0.900     0.889     0.901

--- Original Bootstrap CI ---


early minus text test macro-F1, 95% bootstrap interval [0.041, 0.078]


late minus text test macro-F1, 95% bootstrap interval [0.039, 0.076]

per-class test F1
                 image  text  early   late
informative      0.878  0.88  0.925  0.925
not_informative  0.773  0.78  0.854  0.853



=== humanitarian (agreed labels) train 6126, dev 998, test 955 ===
model     setting  train mF1  dev mF1  test acc  test mF1  test wF1
image       0.001      0.792    0.626     0.781     0.702     0.784
 text       0.001      0.760    0.648     0.758     0.677     0.764
early        0.01      0.981    0.781     0.862     0.794     0.864
 late w_image=0.5      0.877    0.753     0.859     0.802     0.861

--- Original Bootstrap CI ---


early minus text test macro-F1, 95% bootstrap interval [0.071, 0.164]


late minus text test macro-F1, 95% bootstrap interval [0.087, 0.166]

--- NEW: Fusion vs Image-only ---



EARLY vs IMAGE-ONLY
Macro-F1 difference: 0.092
95% Bootstrap CI: [0.039, 0.142]
CI excludes zero: positive improvement



LATE vs IMAGE-ONLY
Macro-F1 difference: 0.101
95% Bootstrap CI: [0.045, 0.157]
CI excludes zero: positive improvement

per-class test F1
                                        image   text  early   late
affected_individuals                    0.452  0.438  0.538  0.615
infrastructure_and_utility_damage       0.775  0.687  0.902  0.864
not_humanitarian                        0.813  0.801  0.887  0.882
other_relevant_information              0.803  0.753  0.854  0.861
rescue_volunteering_or_donation_effort  0.667  0.706  0.789  0.789
